# Pipeline de préparation des données – Bank Marketing (version enrichie)

## Objectif
Construire un pipeline commun de préparation des données (nettoyage, feature engineering, split, preprocessing)
afin de garantir une évaluation cohérente des modèles de classification.

La version enrichie du dataset (Moro et al., 2014) est utilisée.
La variable `duration` est exclue car elle n’est connue qu’après l’appel et introduirait une fuite d’information
dans un cadre prédictif réaliste.


### Chargement des données

In [ ]:
import pandas as pd
import numpy as np

from src.utils_ml import split_data, infer_columns, build_preprocessor

pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42

In [ ]:
DATA_PATH = "data/raw/bank-additional-full.csv"

df = pd.read_csv(DATA_PATH, sep=";")
print("Shape :", df.shape)
df.head()


### Informations générales

In [ ]:
df.info()


In [ ]:
df.describe()


### Traitement des données

In [ ]:
# Transformation de la cible en binaire
df["y"] = df["y"].map({"yes": 1, "no": 0})

df["y"].value_counts(normalize=True).rename("proportion")

In [ ]:
if "duration" in df.columns:
    df = df.drop(columns=["duration"])

df.columns


### Traitement des valeurs manquantes

In [ ]:
# Pas de NaN explicites
df.isnull().sum()


In [ ]:
# Comptage des modalités "unknown"
unknown_counts = (
    df.select_dtypes(include=["object", "category"])
      .apply(lambda s: (s == "unknown").sum())
      .sort_values(ascending=False)
)

unknown_counts


### Feature engineering

In [ ]:
# 1) Binning du nombre de contacts pendant la campagne
df["campaign_bin"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 3, np.inf],
    labels=["1_contact", "2_3_contacts", "4plus_contacts"]
)

# 2) Indicateur de contact précédent
df["has_previous"] = (df["previous"] > 0).astype(int)

# 3) Transformation de pdays (999 = jamais contacté)
df["never_contacted_before"] = (df["pdays"] == 999).astype(int)
df["pdays_clean"] = df["pdays"].replace(999, np.nan)


### Mise à jour des listes de variables

In [ ]:
num_cols, cat_cols = infer_columns(df.drop(columns=["y"]))

num_cols, cat_cols


In [ ]:
split = split_data(
    df,
    target="y",
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=True
)

X_train, X_test = split.X_train, split.X_test
y_train, y_test = split.y_train, split.y_test

X_train.shape, X_test.shape


In [ ]:
preprocessor = build_preprocessor(num_cols, cat_cols)

preprocessor
